# NB10v2: Hierarchical Bayesian Prototype — PyMC Beta-Binomial

In [1]:
import os
os.environ['PYTENSOR_FLAGS'] = 'floatX=float64'
import warnings; warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from sklearn.metrics import brier_score_loss, roc_auc_score, log_loss
from sklearn.preprocessing import StandardScaler

DATA_DIR = '/Users/dlau/repos/fish-welfare/data/'
OUT_DIR  = '/Users/dlau/repos/fish-welfare/data/'
features = ['month_sin', 'doy_cos', 'night_precip_sum', 'precip_2d_sum', 'night_wind_min']
RANDOM_SEED = 42
a0 = 0.3

print(f"PyMC {pm.__version__}, ArviZ {az.__version__}")


PyMC 5.28.0, ArviZ 0.23.4


## 1. Load Data & Priors

In [2]:
# Load prior from NB07v2 or fallback
try:
    prior_df = pd.read_csv(DATA_DIR + 'nb07v2_source_posterior.csv')
    beta_prior_mean = prior_df.set_index('param').loc[features, 'pymc_mean'].values
    beta_prior_std  = prior_df.set_index('param').loc[features, 'pymc_std'].values / np.sqrt(a0)
    alpha_prior_mean = float(prior_df.set_index('param').loc['alpha', 'pymc_mean'])
    print("Loaded priors from nb07v2_source_posterior.csv")
except Exception as e:
    print(f"NB07v2 priors not found ({e}), using MAP fallback")
    src_post = pd.read_csv(DATA_DIR + 'nb07_source_posterior.csv')
    beta_prior_mean = np.array([
        src_post[src_post.param==f]['map_mean'].values[0] for f in features
    ])
    beta_prior_std = np.ones(len(features)) * 2.0 / np.sqrt(a0)
    alpha_prior_mean = float(src_post[src_post.param=='intercept']['map_mean'].values[0])

print(f"Prior means: {dict(zip(features, beta_prior_mean.round(3)))}")
print(f"Prior stds (a0={a0}): {dict(zip(features, beta_prior_std.round(3)))}")

# Load source data
src = pd.read_csv(DATA_DIR + 'nb04_source_features.csv', parse_dates=['date'])
src_c = src[features + ['frac_low', 'n_ponds']].dropna().copy()
src_c['n_total'] = src_c['n_ponds'].astype(int)
src_c['n_low'] = np.minimum((src_c['frac_low'] * src_c['n_total']).round().astype(int), src_c['n_total'])
src_c = src_c[src_c['n_total'] > 0].reset_index(drop=True)

# Load target data
tgt = pd.read_csv(DATA_DIR + 'nb04_target_features.csv', parse_dates=['date']).sort_values('date')
split_date = pd.read_csv(DATA_DIR + 'nb08_split_date.csv')['split_date'].iloc[0]

tgt_c = tgt[features + ['n_total', 'n_low', 'frac_low', 'date']].dropna().copy()
tgt_c['n_total'] = tgt_c['n_total'].astype(int)
tgt_c['n_low']   = np.minimum(tgt_c['n_low'].astype(int), tgt_c['n_total'])
tgt_c = tgt_c[tgt_c['n_total'] > 0].reset_index(drop=True)
tgt_c['bad_day'] = (tgt_c['frac_low'] >= 0.3).astype(int)
tgt_c['season']  = pd.to_datetime(tgt_c['date']).dt.month.map(
    lambda m: 'winter' if m in [12,1,2] else ('monsoon' if m in [6,7,8,9] else 'other'))

train = tgt_c[tgt_c.date < split_date].reset_index(drop=True)
test  = tgt_c[tgt_c.date >= split_date].reset_index(drop=True)

# Standardize features
scaler = StandardScaler()
X_src_std = scaler.fit_transform(src_c[features].values)
X_tgt_all_std = scaler.transform(tgt_c[features].values)

n_tr = len(train); n_te = len(test)
split_idx = n_tr  # index in tgt_c

X_train_std = X_tgt_all_std[:n_tr]
X_test_std  = X_tgt_all_std[n_tr:]

n_src  = src_c['n_total'].values.astype(int)
k_src  = src_c['n_low'].values.astype(int)

n_train = train['n_total'].values.astype(int)
k_train = train['n_low'].values.astype(int)
n_test  = test['n_total'].values.astype(int)
k_test  = test['n_low'].values.astype(int)

print(f"Source: {len(src_c)} obs, Train: {n_tr}, Test: {n_te}")


Loaded priors from nb07v2_source_posterior.csv
Prior means: {'month_sin': np.float64(0.216), 'doy_cos': np.float64(0.282), 'night_precip_sum': np.float64(0.162), 'precip_2d_sum': np.float64(-0.112), 'night_wind_min': np.float64(-0.131)}
Prior stds (a0=0.3): {'month_sin': np.float64(0.149), 'doy_cos': np.float64(0.154), 'night_precip_sum': np.float64(0.179), 'precip_2d_sum': np.float64(0.244), 'night_wind_min': np.float64(0.137)}
Source: 53 obs, Train: 596, Test: 150


## 2. Joint BetaBinomial Model

In [3]:
with pm.Model() as joint_model:
    alpha_src = pm.Normal('alpha_src', mu=alpha_prior_mean, sigma=2)
    alpha_tgt = pm.Normal('alpha_tgt', 0, 5)
    beta = pm.Normal('beta', mu=beta_prior_mean, sigma=beta_prior_std, shape=len(features))
    kappa = pm.Lognormal('kappa', mu=np.log(2), sigma=1)

    # Source likelihood
    eta_src = alpha_src + pm.math.dot(X_src_std, beta)
    mu_src  = pm.math.sigmoid(eta_src)
    a_src   = pm.math.clip(mu_src * kappa, 0.01, 1e4)
    b_src   = pm.math.clip((1 - mu_src) * kappa, 0.01, 1e4)
    y_src   = pm.BetaBinomial('y_src', alpha=a_src, beta=b_src, n=n_src, observed=k_src)

    # Target train likelihood
    eta_tgt = alpha_tgt + pm.math.dot(X_train_std, beta)
    mu_tgt  = pm.math.sigmoid(eta_tgt)
    a_tgt   = pm.math.clip(mu_tgt * kappa, 0.01, 1e4)
    b_tgt   = pm.math.clip((1 - mu_tgt) * kappa, 0.01, 1e4)
    y_tgt   = pm.BetaBinomial('y_tgt', alpha=a_tgt, beta=b_tgt, n=n_train, observed=k_train)

    trace = pm.sample(
        1000, tune=1000, chains=4, target_accept=0.9,
        progressbar=False, random_seed=RANDOM_SEED,
        return_inferencedata=True
    )

print("=== Joint Model MCMC Diagnostics ===")
summary = az.summary(trace, var_names=['alpha_src', 'alpha_tgt', 'beta', 'kappa'])
print(summary.to_string())
print(f"\nMax R-hat:   {summary['r_hat'].max():.4f}")
print(f"Min ESS:     {summary['ess_bulk'].min():.0f}")
print(f"Divergences: {trace.sample_stats['diverging'].sum().item()}")


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [alpha_src, alpha_tgt, beta, kappa]
Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 6 seconds.


=== Joint Model MCMC Diagnostics ===
            mean     sd  hdi_3%  hdi_97%  mcse_mean  mcse_sd  ess_bulk  ess_tail  r_hat
alpha_src -0.088  0.161  -0.397    0.207      0.002    0.003    4347.0    2995.0    1.0
alpha_tgt -2.036  0.137  -2.295   -1.778      0.003    0.002    2791.0    2769.0    1.0
beta[0]   -0.137  0.029  -0.194   -0.085      0.000    0.000    3976.0    3153.0    1.0
beta[1]   -0.010  0.005  -0.019   -0.001      0.000    0.000    2634.0    2766.0    1.0
beta[2]    0.002  0.001  -0.001    0.004      0.000    0.000    3362.0    2830.0    1.0
beta[3]    0.000  0.001  -0.002    0.002      0.000    0.000    2787.0    2568.0    1.0
beta[4]   -0.065  0.046  -0.152    0.020      0.001    0.001    4584.0    2870.0    1.0
kappa      2.069  0.270   1.575    2.577      0.004    0.005    4511.0    2980.0    1.0

Max R-hat:   1.0000
Min ESS:     2634
Divergences: 0


In [4]:
fig, axes = plt.subplots(len(features)+3, 2, figsize=(12, 3*(len(features)+3)))
param_list = ['alpha_src', 'alpha_tgt'] + [f'beta[{i}]' for i in range(len(features))] + ['kappa']

for i, pname in enumerate(param_list):
    for chain_idx in range(4):
        if pname in ['alpha_src', 'alpha_tgt', 'kappa']:
            vals = trace.posterior[pname].values[chain_idx]
        else:
            j = int(pname.split('[')[1].rstrip(']'))
            vals = trace.posterior['beta'].values[chain_idx, :, j]
        axes[i,0].plot(vals, alpha=0.6, lw=0.5)
        axes[i,1].hist(vals, bins=40, alpha=0.5)
    axes[i,0].set_title(f'{pname} trace')
    axes[i,1].set_title(f'{pname} posterior')

plt.tight_layout()
plt.savefig(OUT_DIR + 'nb10v2_trace.png', dpi=80, bbox_inches='tight')
plt.show()


## 3. Test Set Predictions & Evaluation

In [5]:
# Extract posterior samples
beta_samp = trace.posterior['beta'].values.reshape(-1, len(features))
alpha_tgt_samp = trace.posterior['alpha_tgt'].values.flatten()
kappa_samp = trace.posterior['kappa'].values.flatten()

# Posterior predictive for test set
eta_test = alpha_tgt_samp[:, None] + beta_samp @ X_test_std.T
mu_test = 1 / (1 + np.exp(-eta_test))
y_pred_mean  = mu_test.mean(0)
y_pred_lower = np.quantile(mu_test, 0.05, axis=0)
y_pred_upper = np.quantile(mu_test, 0.95, axis=0)

# Evaluate
y_true_bin = test['bad_day'].values
y_true_frac = test['frac_low'].values

if len(np.unique(y_true_bin)) > 1:
    brier = brier_score_loss(y_true_bin, y_pred_mean)
    ll    = log_loss(y_true_bin, y_pred_mean)
    auc   = roc_auc_score(y_true_bin, y_pred_mean)
else:
    brier = brier_score_loss(y_true_bin, y_pred_mean)
    ll = auc = np.nan
    print("Warning: test set has no variance in bad_day")

print(f"=== NB10v2 Test Set Performance ===")
print(f"Brier: {brier:.4f}")
print(f"Log-loss: {ll:.4f}")
print(f"AUC: {auc:.4f}")

# Compare with baselines
baselines = pd.read_csv(DATA_DIR + 'nb09_baseline_results.csv')
print("\nComparison with baselines:")
print(f"{'Model':<25} {'Brier':>8} {'AUC':>8}")
print("-"*45)
for _, row in baselines.iterrows():
    print(f"{row['model']:<25} {row['brier']:>8.4f} {row['auc']:>8.3f}")
print(f"{'NB10v2 BetaBin MCMC':<25} {brier:>8.4f} {auc:>8.4f}")
print()
marginal_brier = baselines[baselines['model'].str.contains('Marginal')]['brier'].values[0]
if brier < marginal_brier:
    print(f"\u2713 NB10v2 BEATS marginal baseline by {marginal_brier - brier:.4f}")
else:
    print(f"\u2717 NB10v2 does NOT beat marginal baseline ({brier:.4f} vs {marginal_brier:.4f})")


=== NB10v2 Test Set Performance ===
Brier: 0.0983
Log-loss: 0.3531
AUC: 0.6964

Comparison with baselines:
Model                        Brier      AUC
---------------------------------------------
B1: Marginal rate           0.0408    0.500
B2: Seasonal rate           0.0686    0.573
B3: Target-only GLM         0.0412    0.646
B4: Pooled GLM              0.0412    0.646
B5: Persistence (prev day)   0.0503    0.424
NB10v2 BetaBin MCMC         0.0983   0.6964

✗ NB10v2 does NOT beat marginal baseline (0.0983 vs 0.0408)


In [6]:
print("=== Beta Posterior 95% Credible Intervals ===")
beta_post = trace.posterior['beta'].values.reshape(-1, len(features))
kappa_post = trace.posterior['kappa'].values.flatten()

print(f"{'Feature':<22} {'Mean':>8} {'2.5%':>8} {'97.5%':>8} {'Non-zero?':>12}")
print("-"*65)
for i, f in enumerate(features):
    b = beta_post[:, i]
    lo, hi = np.quantile(b, [0.025, 0.975])
    non_zero = "YES" if (lo > 0 or hi < 0) else "no"
    print(f"{f:<22} {b.mean():>8.4f} {lo:>8.4f} {hi:>8.4f} {non_zero:>12}")

print(f"\nkappa: mean={kappa_post.mean():.3f}, 95% CI=[{np.quantile(kappa_post,0.025):.3f}, {np.quantile(kappa_post,0.975):.3f}]")
print(f"  kappa > 1 \u2192 overdispersed BetaBinomial: {'YES' if kappa_post.mean() > 1 else 'NO'}")


=== Beta Posterior 95% Credible Intervals ===
Feature                    Mean     2.5%    97.5%    Non-zero?
-----------------------------------------------------------------
month_sin               -0.1373  -0.1955  -0.0810          YES
doy_cos                 -0.0102  -0.0192  -0.0010          YES
night_precip_sum         0.0016  -0.0011   0.0043           no
precip_2d_sum            0.0000  -0.0021   0.0021           no
night_wind_min          -0.0652  -0.1568   0.0261           no

kappa: mean=2.069, 95% CI=[1.587, 2.652]
  kappa > 1 → overdispersed BetaBinomial: YES


## 4. a₀ Sensitivity Analysis

In [7]:
a0_values_sens = [0.0, 0.3, 1.0]
sens_results = []

for a0_v in a0_values_sens:
    if a0_v > 0:
        s_std = beta_prior_std * np.sqrt(a0) / np.sqrt(a0_v)  # rescale from a0=0.3
    else:
        s_std = np.ones(len(features)) * 10.0
    s_std = np.clip(s_std, 0.1, 20.0)

    print(f"\nFitting sensitivity model a0={a0_v}...")
    with pm.Model() as sens_model:
        a_src2 = pm.Normal('alpha_src', mu=alpha_prior_mean, sigma=2)
        a_tgt2 = pm.Normal('alpha_tgt', 0, 5)
        b_s    = pm.Normal('beta', mu=beta_prior_mean, sigma=s_std, shape=len(features))
        k_s    = pm.Lognormal('kappa', mu=np.log(2), sigma=1)

        e_src  = a_src2 + pm.math.dot(X_src_std, b_s)
        m_src  = pm.math.sigmoid(e_src)
        _      = pm.BetaBinomial('y_src', alpha=pm.math.clip(m_src*k_s,0.01,1e4),
                                  beta=pm.math.clip((1-m_src)*k_s,0.01,1e4),
                                  n=n_src, observed=k_src)
        e_tgt  = a_tgt2 + pm.math.dot(X_train_std, b_s)
        m_tgt  = pm.math.sigmoid(e_tgt)
        _      = pm.BetaBinomial('y_tgt', alpha=pm.math.clip(m_tgt*k_s,0.01,1e4),
                                  beta=pm.math.clip((1-m_tgt)*k_s,0.01,1e4),
                                  n=n_train, observed=k_train)

        tr_s = pm.sample(500, tune=500, chains=2, target_accept=0.9,
                         progressbar=False, random_seed=RANDOM_SEED, return_inferencedata=True)

    b_samp_s = tr_s.posterior['beta'].values.reshape(-1, len(features))
    a_tgt_s  = tr_s.posterior['alpha_tgt'].values.flatten()
    eta_te   = a_tgt_s[:,None] + b_samp_s @ X_test_std.T
    mu_te    = 1/(1+np.exp(-eta_te))
    preds    = mu_te.mean(0)
    if len(np.unique(y_true_bin)) > 1:
        b_score = brier_score_loss(y_true_bin, preds)
        auc_s   = roc_auc_score(y_true_bin, preds)
    else:
        b_score = brier_score_loss(y_true_bin, preds)
        auc_s   = np.nan
    sens_results.append({'a0': a0_v, 'brier': b_score, 'auc': auc_s})
    print(f"  a0={a0_v}: Brier={b_score:.4f}, AUC={auc_s:.4f}")

print("\nSensitivity summary:")
print(pd.DataFrame(sens_results).to_string(index=False))



Fitting sensitivity model a0=0.0...


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_src, alpha_tgt, beta, kappa]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 6 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...


  a0=0.0: Brier=0.0996, AUC=0.6970

Fitting sensitivity model a0=0.3...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_src, alpha_tgt, beta, kappa]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 4 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics
Initializing NUTS using jitter+adapt_diag...


  a0=0.3: Brier=0.0982, AUC=0.6959

Fitting sensitivity model a0=1.0...


Multiprocess sampling (2 chains in 2 jobs)
NUTS: [alpha_src, alpha_tgt, beta, kappa]
Sampling 2 chains for 500 tune and 500 draw iterations (1_000 + 1_000 draws total) took 5 seconds.
We recommend running at least 4 chains for robust computation of convergence diagnostics


  a0=1.0: Brier=0.0972, AUC=0.6996

Sensitivity summary:
 a0    brier      auc
0.0 0.099552 0.696954
0.3 0.098222 0.695903
1.0 0.097207 0.699580


## 5. Season-Specific Intercepts

In [8]:
season_map = {'winter': 0, 'monsoon': 1, 'other': 2}
n_seasons = 3
train_seas = train['season'].map(season_map).values.astype(int)

print("Fitting model with season-specific intercepts...")
with pm.Model() as season_model:
    mu_a = pm.Normal('mu_alpha', 0, 2)
    sig_a = pm.HalfNormal('sigma_alpha', 1)
    alpha_seas = pm.Normal('alpha_seas', mu=mu_a, sigma=sig_a, shape=n_seasons)
    beta_s = pm.Normal('beta', mu=beta_prior_mean, sigma=beta_prior_std, shape=len(features))
    kappa_s = pm.Lognormal('kappa', mu=np.log(2), sigma=1)

    eta_s = alpha_seas[train_seas] + pm.math.dot(X_train_std, beta_s)
    mu_s  = pm.math.sigmoid(eta_s)
    _     = pm.BetaBinomial('y_tgt', alpha=pm.math.clip(mu_s*kappa_s,0.01,1e4),
                             beta=pm.math.clip((1-mu_s)*kappa_s,0.01,1e4),
                             n=n_train, observed=k_train)

    trace_seas = pm.sample(500, tune=500, chains=2, target_accept=0.9,
                           progressbar=False, random_seed=RANDOM_SEED, return_inferencedata=True)

# Evaluate
test_seas = test['season'].map(season_map).values.astype(int)
b_samp_seas = trace_seas.posterior['beta'].values.reshape(-1, len(features))
a_seas_samp = trace_seas.posterior['alpha_seas'].values.reshape(-1, n_seasons)
k_seas_samp = trace_seas.posterior['kappa'].values.flatten()

eta_test_seas = a_seas_samp[:, test_seas].T + (b_samp_seas @ X_test_std.T).T
mu_test_seas = 1/(1+np.exp(-eta_test_seas.T))
preds_seas = mu_test_seas.mean(0)

if len(np.unique(y_true_bin)) > 1:
    b_seas = brier_score_loss(y_true_bin, preds_seas)
    auc_seas = roc_auc_score(y_true_bin, preds_seas)
else:
    b_seas = brier_score_loss(y_true_bin, preds_seas)
    auc_seas = np.nan
print(f"Season-intercepts model: Brier={b_seas:.4f}, AUC={auc_seas:.4f}")

# Print season intercepts
for sname, sidx in season_map.items():
    samp = trace_seas.posterior['alpha_seas'].values.reshape(-1, n_seasons)[:, sidx]
    print(f"  alpha_{sname}: {samp.mean():.3f} \u00b1 {samp.std():.3f}")


Initializing NUTS using jitter+adapt_diag...


Fitting model with season-specific intercepts...


ERROR (pytensor.graph.rewriting.basic): SequentialGraphRewriter apply <pytensor.tensor.rewriting.elemwise.InplaceElemwiseOptimizer object at 0x113f7af90>
ERROR (pytensor.graph.rewriting.basic): Traceback:
ERROR (pytensor.graph.rewriting.basic): Traceback (most recent call last):
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/features.py", line 643, in validate_
    ret = fgraph.execute_callbacks("validate")
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/fg.py", line 721, in execute_callbacks
    fn(self, *args, **kwargs)
    ~~^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/dlau/repos/fish-welfare/.venv/lib/python3.14/site-packages/pytensor/graph/destroyhandler.py", line 677, in validate
    raise InconsistencyError("Dependency graph contains cycles")
pytensor.graph.utils.InconsistencyError: Dependency graph contains cycles

During handling of the above exception, another exception occurred:

Traceback (most rec

Season-intercepts model: Brier=0.1055, AUC=0.6686
  alpha_winter: -1.823 ± 0.160
  alpha_monsoon: -1.300 ± 0.482
  alpha_other: -1.913 ± 0.236


## 6. Operational Predictions with 90% CI

In [9]:
# Show predictions with 90% CI for last 30 test days
n_show = min(30, len(test))
fig, ax = plt.subplots(figsize=(12, 4))

x_range = range(n_show)
ax.fill_between(x_range, y_pred_lower[:n_show], y_pred_upper[:n_show],
                alpha=0.3, label='90% CI')
ax.plot(x_range, y_pred_mean[:n_show], label='Posterior mean', lw=2)
ax.scatter(x_range, test['frac_low'].values[:n_show], color='red', s=20, zorder=5, label='Observed frac_low')
ax.axhline(0.3, color='r', linestyle='--', alpha=0.5, label='Threshold (0.3)')
ax.set_xlabel('Test day')
ax.set_ylabel('frac_low (predicted probability)')
ax.set_title(f'Operational Predictions with 90% CI (first {n_show} test days)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR + 'nb10v2_predictions.png', dpi=80, bbox_inches='tight')
plt.show()


## 7. Summary & Go/No-Go Recommendation

In [10]:
print("=" * 60)
print("NB10v2 FINAL SUMMARY")
print("=" * 60)
print(f"\nModel: Joint PyMC Beta-Binomial with power prior (a0={a0})")
print(f"MCMC: 4 chains \u00d7 1000 samples + 1000 tune")
print(f"\nConvergence:")
print(f"  Max R-hat:   {summary['r_hat'].max():.4f}")
print(f"  Min ESS:     {summary['ess_bulk'].min():.0f}")
print(f"  Divergences: {trace.sample_stats['diverging'].sum().item()}")
print(f"\nOverdispersion:")
kappa_mean = kappa_post.mean()
kappa_lo, kappa_hi = np.quantile(kappa_post, [0.025, 0.975])
print(f"  kappa = {kappa_mean:.2f} (95% CI: [{kappa_lo:.2f}, {kappa_hi:.2f}])")
print(f"\nTest set performance:")
print(f"  Brier: {brier:.4f}  (marginal baseline: {marginal_brier:.4f})")
print(f"  AUC:   {auc:.4f}")
print()

if brier < marginal_brier:
    verdict = "GO"
    reason = f"BetaBinomial model beats marginal baseline (Brier {brier:.4f} < {marginal_brier:.4f})"
else:
    verdict = "NO-GO (yet)"
    reason = f"Model doesn't beat marginal baseline. Consider: more target data, stronger priors from larger source, or seasonal sub-models."

print(f"GO/NO-GO: {verdict}")
print(f"Reason: {reason}")

# Save predictions
pred_df = pd.DataFrame({
    'date': test['date'].values,
    'pred_mean': y_pred_mean,
    'pred_lower90': y_pred_lower,
    'pred_upper90': y_pred_upper,
    'observed_frac': y_true_frac,
    'observed_bad': y_true_bin
})
pred_df.to_csv(DATA_DIR + 'nb10v2_predictions.csv', index=False)
print("\nSaved nb10v2_predictions.csv")


NB10v2 FINAL SUMMARY

Model: Joint PyMC Beta-Binomial with power prior (a0=0.3)
MCMC: 4 chains × 1000 samples + 1000 tune

Convergence:
  Max R-hat:   1.0000
  Min ESS:     2634
  Divergences: 0

Overdispersion:
  kappa = 2.07 (95% CI: [1.59, 2.65])

Test set performance:
  Brier: 0.0983  (marginal baseline: 0.0408)
  AUC:   0.6964

GO/NO-GO: NO-GO (yet)
Reason: Model doesn't beat marginal baseline. Consider: more target data, stronger priors from larger source, or seasonal sub-models.

Saved nb10v2_predictions.csv
